# Clase 14C — Mini ChatGPT con RAG usando el dataset real del curso
**Autor:** Josef Rodríguez

Esta clase integra todo lo aprendido:

- chunking
- embeddings
- vector search
- construcción de contexto
- respuesta con modelo generativo

Dataset utilizado:

`https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv`

# 1. Objetivo de la clase

Construir un sistema estilo **mini ChatGPT con RAG** sobre planes de gobierno.

## Flujo final

```text
Dataset real
   ↓
Chunking
   ↓
Embeddings
   ↓
Índice vectorial
   ↓
Consulta del usuario
   ↓
Top-K chunks
   ↓
Prompt
   ↓
LLM
   ↓
Respuesta
```

# 2. Cargar datos y preparar chunks

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv"
df = pd.read_csv(url)
df = df.rename(columns={df.columns[0]: "presidente", df.columns[1]: "texto_plan"})
df["texto_plan"] = df["texto_plan"].fillna("").astype(str)

df.head()

In [ ]:
def chunk_text(text, chunk_size=180, overlap=40):
    words = text.split()
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        part = words[i:i+chunk_size]
        if not part:
            continue
        chunks.append(" ".join(part))
        if i + chunk_size >= len(words):
            break
    return chunks

rows = []
for _, row in df.iterrows():
    chunks = chunk_text(row["texto_plan"], chunk_size=180, overlap=40)
    for j, ch in enumerate(chunks):
        rows.append({
            "presidente": row["presidente"],
            "chunk_id": j,
            "chunk_text": ch
        })

chunks_df = pd.DataFrame(rows)
chunks_df.head()

# 3. Embeddings e índice vectorial

In [ ]:
# Si estás en un entorno limpio, descomenta:
# !pip install -q sentence-transformers faiss-cpu transformers accelerate sentencepiece

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embed_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True
)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

index.ntotal

# 4. Retriever

In [ ]:
def retrieve_chunks(query, top_k=5):
    q_emb = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    out = chunks_df.iloc[indices[0]].copy()
    out["distance_l2"] = distances[0]
    return out[["presidente", "chunk_id", "chunk_text", "distance_l2"]]

retrieve_chunks("propuestas sobre educación pública", top_k=5)

# 5. Prompt builder

En un sistema RAG serio, el prompt debe:

1. delimitar el contexto,
2. restringir la respuesta a la evidencia,
3. dejar explícito qué hacer si no hay soporte.

In [ ]:
def build_prompt(query, retrieved_df):
    context = "\n\n".join(
        [f"[{r.presidente} | chunk {r.chunk_id}] {r.chunk_text}" for r in retrieved_df.itertuples()]
    )
    prompt = f"""Eres un asistente analítico que responde preguntas sobre planes de gobierno presidenciales.
Usa únicamente el contexto proporcionado.
Si la respuesta no se encuentra claramente en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
{context}

Pregunta:
{query}

Respuesta:
"""
    return prompt

In [ ]:
prompt_preview = build_prompt(
    "¿Qué candidatos hablan de salud pública?",
    retrieve_chunks("salud pública, hospitales y atención primaria", top_k=4)
)

print(prompt_preview[:2500])

# 6. Integración con un modelo generativo

Para docencia y pruebas locales, puedes usar modelos pequeños de Hugging Face.  
Dependiendo del entorno, algunos modelos pueden requerir más memoria.

## Opción docente liviana
- `google/flan-t5-base`

## Otras opciones
- modelos instruccionales pequeños
- APIs externas si el curso lo permite

In [ ]:
from transformers import pipeline

# Nota:
# Dependiendo del entorno, podrías preferir cambiar el modelo.
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

# 7. Función final del sistema RAG

In [ ]:
def rag_answer(query, top_k=5, max_new_tokens=220):
    retrieved = retrieve_chunks(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    response = generator(prompt, max_new_tokens=max_new_tokens)[0]["generated_text"]
    return {
        "query": query,
        "retrieved": retrieved,
        "prompt": prompt,
        "answer": response
    }

In [ ]:
result = rag_answer("¿Qué candidatos mencionan educación pública y mejora de la calidad educativa?", top_k=4)
print(result["answer"])

In [ ]:
result["retrieved"][["presidente", "chunk_id", "distance_l2"]]

# 8. Casos de uso sobre el dataset real

Prueba preguntas como:

- ¿Qué candidatos hablan de salud pública?
- ¿Qué propuestas aparecen sobre telemedicina?
- ¿Qué candidatos mencionan minería y crecimiento económico?
- ¿Qué candidatos proponen mejoras en educación básica?
- ¿Qué evidencias concretas recupera el sistema?

Una buena práctica docente es comparar:
1. la respuesta final,
2. los chunks recuperados,
3. la trazabilidad hacia la evidencia.

In [ ]:
preguntas_demo = [
    "¿Qué candidatos hablan de hospitales y salud pública?",
    "¿Qué candidatos mencionan telemedicina?",
    "¿Qué candidatos hablan de minería y exportaciones?",
    "¿Qué candidatos proponen mejoras en educación básica?"
]

for p in preguntas_demo:
    print("="*100)
    print("PREGUNTA:", p)
    out = rag_answer(p, top_k=4, max_new_tokens=180)
    print("RESPUESTA:", out["answer"])
    print()

# 9. Evaluación cualitativa del sistema

Al evaluar un mini chatbot con RAG no basta con leer la respuesta final.  
Debemos revisar:

1. **Retrieval**: ¿recuperó evidencia pertinente?
2. **Grounding**: ¿la respuesta está soportada por el contexto?
3. **Cobertura**: ¿faltó información importante?
4. **Faithfulness**: ¿inventó algo no presente?
5. **Claridad**: ¿respondió de forma útil?

## Posibles mejoras
- reranking
- chunking semántico por párrafos
- filtros por candidato
- ranking híbrido TF-IDF + embeddings
- modelos generativos más fuertes

# 10. Cierre de la masterclass

Con esta clase cerramos el bloque moderno de NLP aplicado:

- En la clase 14A representamos texto con embeddings.
- En la clase 14B construimos retrieval semántico con chunks y FAISS.
- En la clase 14C integramos todo en un mini sistema conversacional con RAG.

## Frase final
**El verdadero valor de un sistema de IA aplicada no está solo en generar texto, sino en generar respuestas sustentadas por evidencia recuperada.**